# BindingDB preparation for **CTDDG conditional fine-tuning**

This notebook creates **fold-preserving, protein-index-aligned, RF-labelled** train/validation/test TSVs. Unlike the older optional split reconstruction, it uses the existing `d1_tr/te_unique_proteins.txt` as authoritative. It also checks the original ProSE/Bepler FASTA and `(6165,)` HDF5 embeddings. It never overwrites raw BindingDB data or regenerates embeddings.

**Ames-positive (1) is predicted mutagenic, Ames-negative (0) is predicted non-mutagenic.** These are Random Forest predictions, **not experimentally measured BindingDB Ames labels**. The RF must be the **same selected model and fingerprint recipe** used to label ChEMBL. The paper instead reports pkCSM for BindingDB; using the RF is an explicitly documented experimental deviation.

**Safety:** run the first checks before setting `RUN_PREPROCESSING=True`. The preprocessor never changes the pretraining atom vocabulary. A legitimate test fold can share ligands with train *on different proteins*, but train/test protein sequences must be disjoint. Run on HPC CPU allocation if processing all five-fold exclusion sets. No source data or old output is modified.


In [7]:
from pathlib import Path
import os, csv, json, hashlib, time, random, collections, shutil, tempfile
import numpy as np
import pandas as pd
import joblib, h5py
from rdkit import Chem, DataStructs, rdBase, RDLogger
RDLogger.DisableLog('rdApp.error')
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator

REPO_ROOT = Path(os.environ.get('CTDDG_REPO_ROOT', '/home/muhamed/repo/CTDDG'))
BINDINGDB_ROOT = REPO_ROOT / 'datasets/data/bindingdb'
FOLD = 'd1'
FOLD_SUFFIX = {'d1':'4_org_1042_104','d2':'4_org_1000_112','d3':'4_org_1004_122',
               'd4':'4_org_1002_103','d5':'4_org_1036_124'}
# Exactly the atom-type file used for your CURRENT pretraining run, not the older 65-type file.
ATOM_TYPES_FILE = REPO_ROOT / 'REWORKED_CODE/chembl_preprocessed/atom_types.txt'
# RF.ipynb saves these inside rf_outputs, depending on where it was executed.
RF_DIR = REPO_ROOT / 'REWORKED_CODE/Random_Forest/rf_outputs'
RF_FILE = RF_DIR / 'best_ames_rf.joblib'
RF_METADATA_FILE = RF_DIR / 'best_ames_rf_metadata.json'
PRETRAIN_CHEMBL_FILE = REPO_ROOT / 'REWORKED_CODE/chembl_preprocessed/chembl_final.txt'
OUTPUT_DIR = REPO_ROOT / 'datasets/processed/ctddg_bindingdb_rf_d1_v1'
RUN_PREPROCESSING = True     # Change to True ONLY after checking paths and settings
CHECK_PRETRAIN_LEAKAGE = True # Direct set-intersection against the EXACT stage-1 molecules
BUILD_ALL_FOLD_EXCLUSION = True # canonical ligands from all five folds; relatively CPU intensive
REQUIRE_EMBEDDINGS = True   # Fail rather than falsely assume protein vectors exist
N_C = 6165
MAX_ATOMS, MAX_STEPS = 100, 120
VAL_PROTEIN_FRACTION, SPLIT_SEED = 0.05, 42
PREDICT_BATCH = 8192
# Output folder is immutable once created: change version/suffix for a new experiment.
print('RDKit:',rdBase.rdkitVersion)
for k,v in {'raw_bindingdb':BINDINGDB_ROOT,'atom_vocab':ATOM_TYPES_FILE,
            'RF':RF_FILE,'RF_metadata':RF_METADATA_FILE,'pretrain_data':PRETRAIN_CHEMBL_FILE,
            'output':OUTPUT_DIR}.items(): print(f'{k:15s}',v,'exists:',v.exists())
assert FOLD in FOLD_SUFFIX


RDKit: 2026.03.6
raw_bindingdb   /home/muhamed/repo/CTDDG/datasets/data/bindingdb exists: True
atom_vocab      /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt exists: True
RF              /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/rf_outputs/best_ames_rf.joblib exists: True
RF_metadata     /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/rf_outputs/best_ames_rf_metadata.json exists: True
pretrain_data   /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/chembl_final.txt exists: True
output          /home/muhamed/repo/CTDDG/datasets/processed/ctddg_bindingdb_rf_d1_v1 exists: False


## 1. Strict file discovery and alignment

Ligand and protein files are **paired line-for-line**. Their lines may contain *space-separated characters*. We reassemble each **whole line** with `''.join(line.split())` before parsing it as a molecule/protein. **Never use `line.split()[0]` for BindingDB ligands.** The per-fold protein lists and FASTA are *not* sorted/reconstructed: the original order determines `p1`, `p2`, etc. in ProSE HDF5.


In [8]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()

def fold_dir(fold,part):
    return BINDINGDB_ROOT / f'{part}_dataset' / f'{part}_{FOLD_SUFFIX[fold]}'

def find_pair_files(fold,part):
    d=fold_dir(fold,part)
    files={}
    for kind in ('ligands','proteins'):
        # Both naming variants appear in the old preprocessors.
        options=[d/f'{kind}_{part}_4_org_corrected', d/f'{kind}_{FOLD_SUFFIX[fold]}_corrected']
        found=[x for x in options if x.is_file()]
        if not found:
            found=sorted(x for x in d.glob(f'{kind}*_corrected') if x.is_file())
        if len(found)!=1:
            raise RuntimeError(f'{kind} in {d}: expected one matching file, found {found}')
        files[kind]=found[0]
    return files

def protein_paths(fold,part):
    tag='tr' if part=='train' else 'te'; d=fold_dir(fold,part)
    return {'list':d/f'{fold}_{tag}_unique_proteins.txt',
            'fasta':d/f'{fold}_{tag}_unique_proteins_bepler.fa',
            'h5':d/f'{fold}_{tag}_unique_embeddings_bepler.h5'}

def spaced_lines(path):
    with open(path) as f:
        for line_no,line in enumerate(f,1):
            v=''.join(line.split())
            if not v: raise ValueError(f'Blank line at {path}:{line_no}; pair alignment at risk')
            yield v

def read_fasta(path):
    records=[]; name=None; seq=[]
    with open(path) as f:
        for line in f:
            line=line.strip()
            if line.startswith('>'):
                if name is not None: records.append((name,''.join(seq).upper()))
                name=line[1:].split()[0];seq=[]
            elif line: seq.append(''.join(line.split()))
    if name is not None: records.append((name,''.join(seq).upper()))
    return records

def verify_protein_order(fold,part):
    files=protein_paths(fold,part)
    for key in ('list','fasta'):
        if not files[key].is_file():raise FileNotFoundError(files[key])
    sequences=[s.upper() for s in spaced_lines(files['list'])]
    if len(sequences)!=len(set(sequences)):
        raise ValueError(f'{files["list"]}: duplicated sequences make mapping ambiguous')
    fasta=read_fasta(files['fasta'])
    if len(fasta)!=len(sequences):raise ValueError(f'FASTA/list count mismatch for {fold}/{part}')
    for i,((name,seq),actual) in enumerate(zip(fasta,sequences),1):
        if name!=f'p{i}' or seq!=actual:
            raise ValueError(f'{fold}/{part}: FASTA/list disagree at index {i}, id {name!r}')
    if REQUIRE_EMBEDDINGS:
        if not files['h5'].is_file():
            raise FileNotFoundError(f'Missing ProSE embedding {files["h5"]}; run 01_generate_prose_embeddings.ipynb with --pool avg')
        with h5py.File(files['h5'],'r') as h:
            if set(h.keys())!={f'p{i}' for i in range(1,len(sequences)+1)}:
                raise ValueError('HDF5 keys must be exactly p1..pN (same FASTA order)')
            for i in range(1,len(sequences)+1):
                x=np.asarray(h[f'p{i}'])
                if x.shape!=(N_C,) or not np.isfinite(x).all():
                    raise ValueError(f'{files["h5"]}: p{i} shape {x.shape} or non-finite; expected ({N_C},)')
    return sequences, files

def read_paired(fold,part):
    f=find_pair_files(fold,part)
    seen=set(); records=[]
    import itertools
    from itertools import zip_longest
    with open(f['ligands']) as ls,open(f['proteins']) as ps:
        for i,pair in enumerate(zip_longest(ls,ps),1):
            ligand,protein=pair
            if ligand is None or protein is None:
                raise ValueError(f'Ligand/protein row counts mismatch: {f} at line {i}')
            smi,seq=''.join(ligand.split()),''.join(protein.split()).upper()
            if not smi or not seq:raise ValueError(f'Empty ligand/protein: {f} at line {i}')
            key=(smi,seq)
            if key not in seen:
                records.append((smi,seq,part,i));seen.add(key)
    return records, {k:str(v) for k,v in f.items()}


## 2. Molecule cleaning and checkpoint vocabulary

The cleaning policy matches the **heavy-atom principal-fragment** choice in `RF.ipynb`. RDKit canonicalization happens before RF labelling and ligand equality checks. Disconnected molecules do not enter graph traversal. `atom_types.txt` is an **ordered checkpoint contract**: OOV ligands are rejected and never added to the vocabulary.


In [9]:
BONDS={Chem.BondType.AROMATIC,Chem.BondType.SINGLE,Chem.BondType.DOUBLE,Chem.BondType.TRIPLE}

def load_vocab():
    if not ATOM_TYPES_FILE.is_file():raise FileNotFoundError(ATOM_TYPES_FILE)
    rows=[]
    for ln,line in enumerate(ATOM_TYPES_FILE.read_text().splitlines(),1):
        if not line.strip():continue
        x=line.strip().split(',')
        if len(x)!=3:raise ValueError(f'Bad atom type at line {ln}')
        rows.append((x[0],int(x[1]),int(x[2])))
    if len(rows)!=len(set(rows)) or not rows:raise ValueError('Duplicate/empty atom vocabulary')
    return set(rows)

def clean_smiles(smi,allowed_atoms=None):
    # Invalid BindingDB records must be reported and skipped, not crash a large
    # CPU preprocessing run after hundreds of thousands of valid records.
    try:
        mol=Chem.MolFromSmiles(smi)
        if mol is None or not mol.GetNumAtoms():return None,'invalid_smiles'
        parts=Chem.GetMolFrags(mol,asMols=True,sanitizeFrags=True)
        if len(parts)>1:
            mol=max(parts,key=lambda x:(x.GetNumHeavyAtoms(),x.GetNumAtoms(),Chem.MolToSmiles(x)))
        if mol.GetNumBonds()==0:return None,'zero_bonds'
        if mol.GetNumAtoms()>MAX_ATOMS:return None,'too_many_atoms'
        if mol.GetNumBonds()+1>MAX_STEPS:return None,'too_many_actions'
        if any(b.GetBondType() not in BONDS for b in mol.GetBonds()):return None,'unsupported_bond'
        if allowed_atoms is not None:
            if any((a.GetSymbol(),a.GetFormalCharge(),a.GetNumExplicitHs()) not in allowed_atoms
                   for a in mol.GetAtoms()):return None,'out_of_vocab'
        return Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True),None
    except Exception:
        return None,'rdkit_exception'

# Two important parser / chemistry regression tests
assert ''.join('C C ( = O ) O'.split())=='CC(=O)O'
assert clean_smiles('CC(=O)Oc1ccccc1C(=O)O.Cl')[0] == Chem.MolToSmiles(Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O'))
print('PASS: whitespace reconstruction and principal-fragment tests')


PASS: whitespace reconstruction and principal-fragment tests


## 3. Label with **exact selected RF weights**

Reads `best_ames_rf.joblib` plus its `best_ames_rf_metadata.json` from **your new RF notebook**, rather than the older `ames_random_forest.pkl` referred to in historical fine-tuning code. The metadata fixes the feature type/radius/width. The RF is never retrained automatically. Only load joblib files you trust.


In [10]:
def load_selected_rf():
    if not RF_FILE.is_file() or not RF_METADATA_FILE.is_file():
        raise FileNotFoundError(f'Need BOTH {RF_FILE} and {RF_METADATA_FILE}. Check RF.ipynb OUTPUT_DIR.')
    rf=joblib.load(RF_FILE) # trusted own file, pickle-based
    meta=json.loads(RF_METADATA_FILE.read_text())
    cfg=meta.get('selected_fingerprint')
    if not isinstance(cfg,dict):raise ValueError('RF metadata missing selected_fingerprint')
    kind=cfg.get('kind')
    if kind not in ('morgan','maccs'):raise ValueError(f'Unsupported RF fingerprint kind: {kind}')
    n_bits=int(cfg.get('n_bits',167 if kind=='maccs' else 2048))
    radius=int(cfg.get('radius',2))
    if kind=='maccs' and n_bits!=167:raise ValueError('RDKit MACCS feature vector must have 167 bits')
    if getattr(rf,'n_features_in_',None)!=n_bits:
        raise ValueError(f'RF expects {getattr(rf,"n_features_in_",None)} features but recipe says {n_bits}')
    if set(map(int,rf.classes_))!={0,1}:raise ValueError(f'RF class encoding {rf.classes_} is not 0/1')
    fp={'kind':kind,'n_bits':n_bits,'radius':radius}
    print('Selected RF:',meta.get('selected_model'), '| features:',fp,'| weights SHA:',sha256(RF_FILE)[:16])
    return rf,fp,meta

def fingerprint_matrix(smiles,recipe):
    n_bits=recipe['n_bits'];a=np.zeros((len(smiles),n_bits),dtype=np.uint8)
    gen=(rdFingerprintGenerator.GetMorganGenerator(radius=recipe['radius'],fpSize=n_bits)
         if recipe['kind']=='morgan' else None)
    for i,s in enumerate(smiles):
        mol=Chem.MolFromSmiles(s)
        if mol is None:raise ValueError(f'Previously cleaned SMILES invalid: {s}')
        bitvect=gen.GetFingerprint(mol) if gen is not None else MACCSkeys.GenMACCSKeys(mol)
        DataStructs.ConvertToNumpyArray(bitvect,a[i])
    return a

def label_unique(unique_smiles):
    rf,recipe,meta=load_selected_rf()
    labels={}
    smi=sorted(set(unique_smiles))
    for off in range(0,len(smi),PREDICT_BATCH):
        chunk=smi[off:off+PREDICT_BATCH]
        pred=rf.predict(fingerprint_matrix(chunk,recipe))
        if not set(np.unique(pred)).issubset({0,1}):raise ValueError('RF output outside 0/1')
        labels.update((s,int(y)) for s,y in zip(chunk,pred))
        if (off//PREDICT_BATCH)%20==0:print(f'  RF: {min(off+len(chunk),len(smi)):,}/{len(smi):,} unique ligands')
    return labels,recipe


## 4. Assemble fold, make protein-level validation, verify leakage

The paired raw files can include proteins outside their designated folds. We route records according to the authoritative train/test **sequence lists**; we pool both raw pair files to retain all associations for their corresponding proteins. Duplicate (ligand, protein) pairs are removed, but the same ligand may correctly bind more than one protein. We do not use any held-out protein for fine-tuning or model selection.


In [11]:
def assemble_fold():
    seq_train,files_train=verify_protein_order(FOLD,'train')
    seq_test,files_test=verify_protein_order(FOLD,'test')
    if set(seq_train)&set(seq_test):raise ValueError('Train/test protein sequence overlap')
    maps={'train':{s:i for i,s in enumerate(seq_train)},'test':{s:i for i,s in enumerate(seq_test)}}
    gathered=[];raw_files={}
    for part in ('train','test'):
        pairs,paths=read_paired(FOLD,part)
        raw_files.update({f'{part}_{kind}':p for kind,p in paths.items()})
        gathered+=pairs
        print(part,'raw unique pairs:',len(pairs))
    seen=set();rows={'train':[],'test':[]};stats=collections.Counter()
    allowed=load_vocab()
    for raw,seq,source,line_no in gathered:
        partitions=[part for part in ('train','test') if seq in maps[part]]
        if not partitions:
            stats['outside_fold_proteins']+=1;continue
        if len(partitions)!=1:raise ValueError('Same protein in two partitions')
        part=partitions[0]
        clean,why=clean_smiles(raw,allowed)
        if why: stats[f'{part}_{why}']+=1;continue
        key=(part,seq,clean)
        if key in seen:stats['duplicate_pair']+=1;continue
        seen.add(key)
        rows[part].append({'smiles':clean,'protein_index':maps[part][seq],
                           'protein_id':f'p{maps[part][seq]+1}',
                           'source_partition':source,'source_line':line_no})
    for part,seqs in [('train',seq_train),('test',seq_test)]:
        have={r['protein_index'] for r in rows[part]}
        missing=set(range(len(seqs)))-have
        if missing:
            raise RuntimeError(f'{part}: {len(missing)} fold proteins have no usable ligand; first IDs {[f"p{i+1}" for i in sorted(missing)[:8]]}. Investigate filtering/vocabulary before training.')
        print(part, len(rows[part]),'pairs;',len(have),'proteins')
    rng=np.random.default_rng(SPLIT_SEED)
    n_val=max(1,round(VAL_PROTEIN_FRACTION*len(seq_train)))
    val_indices=set(map(int,rng.permutation(len(seq_train))[:n_val]))
    train=[r for r in rows['train'] if r['protein_index'] not in val_indices]
    val=[r for r in rows['train'] if r['protein_index'] in val_indices]
    test=rows['test']
    if not train or not val or not test:raise RuntimeError('Empty train, val or test split')
    for group in (train,val,test):
        if any(r['protein_id']!=f'p{r["protein_index"]+1}' for r in group):raise AssertionError('Protein index error')
    if {r['protein_index'] for r in train}&{r['protein_index'] for r in val}:
        raise AssertionError('Train/val share proteins')
    overlap=len({r['smiles'] for r in train}|{r['smiles'] for r in val})
    print('Validation protein rows:',len(val_indices),'ligand/protein pool unique ligands:',overlap)
    return {'train':train,'val':val,'test':test}, {'train':files_train,'test':files_test}, raw_files, dict(stats), sorted(val_indices)

def all_fold_ligands():
    result=set();counts=collections.Counter();paths={}
    for fold in FOLD_SUFFIX:
        for part in ('train','test'):
            p=find_pair_files(fold,part)['ligands'];paths[f'{fold}_{part}']=str(p)
            for raw in spaced_lines(p):
                smi,err=clean_smiles(raw,allowed_atoms=None)
                if smi: result.add(smi)
                else:counts[err]+=1
    print('All-fold canonical ligands:',len(result),'rejected:',dict(counts))
    return result,paths

def pretrain_overlap(exclusion):
    if not PRETRAIN_CHEMBL_FILE.is_file():
        return {'status':'UNVERIFIED','reason':f'exact pretraining input missing at {PRETRAIN_CHEMBL_FILE}'}
    n=0;bad=0;examples=[];seen=set()
    with open(PRETRAIN_CHEMBL_FILE) as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            smi=line.strip().rsplit(maxsplit=1)[0] # pretraining '<SMILES> <class>'
            normalized,err=clean_smiles(smi,allowed_atoms=None)
            if err:bad+=1;continue
            if normalized in exclusion and normalized not in seen:
                n+=1;seen.add(normalized)
                if len(examples)<10:examples.append(normalized)
    return {'status':'verified_zero' if n==0 and bad==0 else 'FAILED',
            'overlapping_unique_ligands':n,'malformed_or_excluded_pretrain_rows':bad,
            'examples':examples,'pretrain_file':str(PRETRAIN_CHEMBL_FILE),
            'pretrain_sha256':sha256(PRETRAIN_CHEMBL_FILE)}


## 5. Write versioned artifacts atomically

The output is **new** (never write over a previous training dataset). The manifest contains hashes for the exact raw paired data, FASTA, HDF5 embeddings, RF weights/metadata, atom vocabulary and produced TSVs, so fine-tuning can refuse changed dependencies. `pretrain_leakage.status` is `verified_zero` **only when a direct full-molecule comparison passes**. All-fold filtering is recommended for paper-quality claims.


In [12]:
def prepare():
    if OUTPUT_DIR.exists():
        raise FileExistsError(f'{OUTPUT_DIR} already exists. Use a NEW versioned output directory; do not silently overwrite a run dataset.')
    if not BUILD_ALL_FOLD_EXCLUSION and CHECK_PRETRAIN_LEAKAGE:
        raise ValueError('A claimed all-fold leakage check requires BUILD_ALL_FOLD_EXCLUSION=True')
    rows,split_files,raw_files,filters,val_indices=assemble_fold()
    unique={r['smiles'] for group in rows.values() for r in group}
    labels,recipe=label_unique(unique)
    for group in rows.values():
        for row in group:row['ames']=labels[row['smiles']]
    exclusion,extra_paths=all_fold_ligands() if BUILD_ALL_FOLD_EXCLUSION else (None,{})
    leakage=(pretrain_overlap(exclusion) if CHECK_PRETRAIN_LEAKAGE else
             {'status':'UNVERIFIED','reason':'direct leakage check disabled'})
    if leakage['status']=='FAILED':
        print('[BLOCKER] Stage-1 ChEMBL data may contain BindingDB ligands: ',leakage)
    if leakage['status']=='UNVERIFIED':
        print('[BLOCKER] Stage-1 anti-leakage provenance is not verified:',leakage)
    OUTPUT_DIR.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(tempfile.mkdtemp(prefix='.bindingdb_tmp_',dir=OUTPUT_DIR.parent))
    try:
        for split in ('train','val','test'):
            pd.DataFrame(rows[split],columns=['smiles','protein_index','protein_id','ames','source_partition','source_line']).to_csv(
                tmp/f'{split}.tsv',sep='	',index=False)
        if exclusion is not None:
            with (tmp/'all_bindingdb_ligands.smi').open('w') as f:
                for smi in sorted(exclusion):f.write(smi+'\n')
        dependencies=[ATOM_TYPES_FILE,RF_FILE,RF_METADATA_FILE]
        for files in split_files.values():dependencies+=list(files.values())
        dependencies+=[Path(x) for x in raw_files.values()]
        dependencies+=[Path(x) for x in extra_paths.values()]
        files_hash={str(path):sha256(path) for path in sorted(set(dependencies))}
        counts={split:{'pairs':len(rows[split]),'unique_proteins':len({r['protein_index'] for r in rows[split]}),
                       'unique_ligands':len({r['smiles'] for r in rows[split]}),
                       'ames_positive':sum(r['ames'] for r in rows[split])} for split in rows}
        manifest={'schema_version':1,'fold':FOLD,'source':'Grechishnikova BindingDB folds',
                  'label_meaning':'RF-predicted Ames positive (1) vs negative (0), not measured',
                  'paper_deviation':'BindingDB Ames labels are selected RF predictions; CTDDG paper used pkCSM',
                  'rf_model':str(RF_FILE),'rf_recipe':recipe,'atom_types_file':str(ATOM_TYPES_FILE),
                  'protein_paths':{split:{k:str(v) for k,v in files.items()} for split,files in split_files.items()},
                  'protein_rows_val':val_indices,'val_fraction':VAL_PROTEIN_FRACTION,'split_seed':SPLIT_SEED,
                  'max_atoms':MAX_ATOMS,'max_steps':MAX_STEPS,'fragment_policy':'max heavy atoms then atom count then canonical smiles',
                  'pool_pair_files':True,'pair_dedup':'protein + canonical ligand',
                  'stats':counts,'filter_counts':filters,'pretrain_leakage':leakage,
                  'dependencies_sha256':files_hash,
                  'output_sha256':{f'{s}.tsv':sha256(tmp/f'{s}.tsv') for s in rows}}
        if exclusion is not None:manifest['output_sha256']['all_bindingdb_ligands.smi']=sha256(tmp/'all_bindingdb_ligands.smi')
        (tmp/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
        os.rename(tmp,OUTPUT_DIR)
    except Exception:
        shutil.rmtree(tmp,ignore_errors=True);raise
    print('\nDATASET READY:',OUTPUT_DIR)
    print(json.dumps({'stats':counts,'leakage':leakage},indent=2))
    print('Do not start fine-tuning if leakage is FAILED/UNVERIFIED or RF provenance is unresolved.')
    return manifest

if RUN_PREPROCESSING:
    manifest=prepare()
else:
    print('DRY MODE. Inspect configuration, then set RUN_PREPROCESSING=True to write the dataset.')


train raw unique pairs: 183964
test raw unique pairs: 9203


RuntimeError: train: 2 fold proteins have no usable ligand; first IDs ['p69', 'p517']. Investigate filtering/vocabulary before training.

## Outputs and next step

- `train.tsv`, `val.tsv`, `test.tsv` — canonical ligand SMILES, original ProSE `protein_index`, `protein_id`, RF Ames label.
- `dataset_manifest.json` — source/feature/model/split hashes, rejection counts and leakage status.
- `all_bindingdb_ligands.smi` — all five folds, for leakage audit when enabled.

The separate **fine-tuning notebook** loads only `train.tsv` and `val.tsv` for optimisation and model selection. The held-out `test.tsv` is reserved for *final* evaluation. It uses the original HDF5 files and the same ordered atom vocabulary as the pretraining checkpoint. **No actual data can be produced here because the BindingDB datasets and trained RF weights are on your HPC server, not among the uploaded files.**
